# 02 — Forecasting Data Preparation

Fresh-kernel reconstruction, temporal sealing, and forecasting-specific handoff for `datasets::nottem`. No model is trained, scored, selected, or compared.

## 1. Preparation Context and Temporal Handoff Boundary

The persisted Notebook 01 exploration handoff is the sole upstream scientific authority. The 1939 final holdout is prospectively sealed from all new learned or selection decisions.

In [16]:
from __future__ import annotations
import json
from pathlib import Path
import pandas as pd
from scripts.download_data import acquire_rdataset
from scripts.forecasting_exploration_handoff import load_and_validate_forecasting_exploration_handoff
from scripts.forecasting_preparation import (build_forecasting_preparation_handoff, load_and_validate_forecasting_preparation_handoff, materialize_temporal_scopes, reconstruct_backtesting_schedule, reconstruct_monthly_series, validate_exploration_readiness, validate_forecasting_source, validate_target_contract, write_forecasting_preparation_artifacts)
from scripts.prepare_data import fingerprint_file
from scripts.project_context import get_project_context
PROJECT = get_project_context()
DATASET_SLUG = 'nottem'
EXPLORATION_PATH = Path('artifacts/exploration/nottem/exploration-handoff.json')
PREPARATION_PATH = Path('artifacts/preparation/nottem/preparation-handoff.json')

## 2. Independent Exploration-Handoff Loading and Readiness Gate

The official validator authenticates schema and forecasting semantics; additional preparation gates fail closed.

In [17]:
exploration_file = PROJECT.require_file(EXPLORATION_PATH)
exploration = load_and_validate_forecasting_exploration_handoff(exploration_file, expected_dataset_slug='nottem', expected_source_reference='datasets::nottem')
validate_exploration_readiness(exploration)
EXPLORATION_SHA256 = fingerprint_file(exploration_file)
source_contract = exploration['source']
prediction_contract = exploration['prediction_contract']
assert exploration['readiness']['model_selection_ready'] is False

## 3. Independent R Dataset Acquisition

The project API reacquires or safely reuses all three raw Rdatasets files without depending on a live Notebook 01 dataframe.

In [18]:
acquisition = acquire_rdataset(dataset_name=source_contract['dataset_name'], package=source_contract['package'], destination=PROJECT.path('data', 'raw', DATASET_SLUG), project_root=PROJECT.root)
raw_path = acquisition.require_one_file('dataset.csv')
metadata_path = acquisition.require_one_file('metadata.json')
acquisition.require_one_file('documentation.txt')
raw = pd.read_csv(raw_path)
source_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))

## 4. Source Identity and Canonical Monthly Series Revalidation

Raw bytes, metadata identity, exact schema, fractional-year month coordinates, continuity, uniqueness, numeric values, and frozen coverage are independently checked.

In [19]:
source_validation = validate_forecasting_source(raw, source_metadata, source_file=raw_path, handoff=exploration, project_root=PROJECT.root)
canonical = reconstruct_monthly_series(raw, target_name='temperature', tolerance=1e-8)
temporal = exploration['temporal_contract']
assert (str(canonical.index[0]), str(canonical.index[-1]), len(canonical)) == (temporal['source_start'], temporal['source_end'], temporal['source_observations'])

## 5. Univariate Forecasting Target Contract Revalidation

The target remains one numeric endogenous monthly series in degrees Fahrenheit, with no target classes or exogenous predictors.

In [20]:
revalidated_prediction_contract = validate_target_contract(canonical, exploration)
assert revalidated_prediction_contract['target_classes'] == []
assert revalidated_prediction_contract['source_exogenous_predictors'] == 0

## 6. Deterministic Prepared-Series Projection

Preparation is preservation: values changed = 0; rows removed/synthesized = 0; missing values imputed = 0. No interpolation, clipping, winsorization, generic outlier treatment, scaling, normalization, differencing, decomposition, detrending, seasonal adjustment, or power transformation is applied.

In [21]:
PRESERVATION = {'values_changed': 0, 'rows_removed': 0, 'rows_synthesized': 0, 'missing_values_imputed': 0, 'interpolation_applied': False, 'clipping_applied': False, 'winsorization_applied': False, 'generic_outlier_treatment': False, 'global_scaling': False, 'global_normalization': False, 'global_differencing': False, 'global_seasonal_differencing': False, 'global_decomposition': False, 'global_detrending': False, 'global_seasonal_adjustment': False, 'global_power_transformation': False, 'source_exogenous_predictors_added': 0}
assert all(value in (0, False) for value in PRESERVATION.values())

## 7. Development Scope and Prospectively Sealed Final Holdout

Only sealing metadata is displayed; no 1939 values, summaries, plots, or scores are exposed.

In [22]:
development, sealed_holdout = materialize_temporal_scopes(canonical, exploration)
holdout_sealing = {'start': temporal['final_holdout_start'], 'end': temporal['final_holdout_end'], 'count': len(sealed_holdout), 'sha256': __import__('scripts.prepare_data', fromlist=['fingerprint_dataframe_csv']).fingerprint_dataframe_csv(sealed_holdout), 'sealed': True, 'evaluated': False}
print(holdout_sealing)

{'start': '1939-01', 'end': '1939-12', 'count': 12, 'sha256': '4379a7e8dad2879ca90ff9137707920e2e6a64095ece8b8587ad9da972aaf0c3', 'sealed': True, 'evaluated': False}


## 8. Expanding-Window Backtesting Schedule Reconstruction

The nine-fold schedule is recomputed from frozen parameters rather than copied from JSON.

In [23]:
development_series = canonical.loc[temporal['development_start']:temporal['development_end']]
schedule = reconstruct_backtesting_schedule(development_series, exploration)
assert len(schedule) == 9 and sum(row['validation_observations'] for row in schedule) == 108

## 9. Fold Integrity and Temporal-Causality Validation

Every training window ends at its origin; every validation starts one month later, windows do not overlap, and no fold reaches 1939. Centered/future-aware rolling, unshifted lags, full-series STL, random splitting, and shuffle are prohibited.

In [24]:
validation_months = []
for row in schedule:
    origin = pd.Period(row['train_end_forecast_origin'], freq='M')
    start = pd.Period(row['validation_start'], freq='M')
    end = pd.Period(row['validation_end'], freq='M')
    assert start == origin + 1 and end < pd.Period('1939-01', freq='M')
    validation_months.extend(pd.period_range(start, end, freq='M'))
assert len(validation_months) == len(set(validation_months)) == 108

## 10. Fold-Local Seasonal MASE Scaling

Each seasonal MASE(12) denominator is computed strictly from that fold's training history and compared with the authenticated upstream schedule. No forecast error is calculated here.

In [25]:
mase_scales = {row['fold']: row['seasonal_mase_scale_from_training'] for row in schedule}
assert all(pd.notna(value) and value > 0 for value in mase_scales.values())

## 11. Frozen Preparation and Leakage-Control Contract

Future learned operations must be fitted independently inside each training fold. Backward-looking lags must use known history only. The final holdout cannot enter model development artifacts or decisions.

In [26]:
evaluation = exploration['evaluation_contract']
assert evaluation['primary_metric'] == 'mae'
assert evaluation['secondary_metrics'] == ['rmse', 'seasonal_mase_12']
assert evaluation['primary_baseline'] == 'seasonal_naive_12' and evaluation['secondary_baseline'] == 'naive_last_value'

## 12. Forecasting Preparation Artifact Contract

The forecasting-specific schema freezes lineage, source identity, explicit CSV references, folds, metrics, baselines, preservation rules, consumer constraints, and readiness—never holdout values.

In [27]:
preparation_payload = build_forecasting_preparation_handoff(handoff=exploration, exploration_handoff_path=EXPLORATION_PATH, exploration_sha256=EXPLORATION_SHA256, source_validation=source_validation, development=development, final_holdout=sealed_holdout, schedule=schedule)
assert preparation_payload['schema_version'] == 'forecasting-preparation-handoff.v1'
assert preparation_payload['readiness']['model_selection_ready'] is True

## 13. Atomic Artifact Materialization

One staged transaction creates artifacts, reuses byte-equivalent artifacts, and rejects divergent existing artifacts without overwrite.

In [28]:
write_result = write_forecasting_preparation_artifacts(project_root=PROJECT.root, development=development, final_holdout=sealed_holdout, payload=preparation_payload)
print(dict(write_result.statuses))

{'artifacts/preparation/nottem/preparation-handoff.json': 'reused_equivalent', 'data/processed/nottem/development.csv': 'reused_equivalent', 'data/processed/nottem/final-holdout.csv': 'reused_equivalent'}


## 14. Preparation Readiness

Model selection becomes ready only after source, reconstruction, preservation, sealing, fold, MASE, leakage, persistence, and reload gates pass. No model has been selected or trained.

In [29]:
readiness = preparation_payload['readiness']
assert readiness['model_selection_ready'] is True
assert readiness['final_holdout_evaluated'] is False
assert readiness['split_execution_ready'] is False
assert readiness['model_selected'] is False and readiness['final_model_trained'] is False

## 15. Fresh-Kernel Handoff Reload and Notebook 03 Boundary

The safe loader authenticates upstream, development, and sealed-holdout hashes, but returns development and contracts only. Notebook 03 must use the frozen folds and keep 1939 closed.

In [30]:
reloaded = load_and_validate_forecasting_preparation_handoff(PREPARATION_PATH, project_root=PROJECT.root, expected_dataset_slug='nottem')
assert len(reloaded.development) == 228
assert not hasattr(reloaded, 'final_holdout')
assert reloaded.sealed_holdout_integrity['sealed'] is True and reloaded.sealed_holdout_integrity['evaluated'] is False
print('Forecasting preparation handoff reload: PASSED; Notebook 03 boundary: SAFE')

Forecasting preparation handoff reload: PASSED; Notebook 03 boundary: SAFE
